In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor
import joblib


data = pd.read_csv('Cleaned_data_for_model.csv')
data.drop(columns=['Unnamed: 0'], inplace=True)


x = data.drop(columns=['price'])
y = data['price']
y_log = np.log(y)

categorical_features = ['property_type', 'location', 'city', 'purpose']
numerical_features = ['bedrooms', 'baths', 'Area_in_Marla']

preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
], remainder='passthrough')

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(objective='reg:squarederror', random_state=42))
])

X_train, X_test, Y_train, Y_test = train_test_split(x, y_log, test_size=0.2, random_state=42)


model_pipeline.fit(X_train, Y_train)

y_pred_log = model_pipeline.predict(X_test)
mae = mean_absolute_error(Y_test, y_pred_log)
mse = mean_squared_error(Y_test, y_pred_log)
rmse = np.sqrt(mse)
r2 = r2_score(Y_test, y_pred_log)

print("Mean Absolute Error:", mae)
print("Mean Squared Error:", mse)
print("Root Mean Squared Error:", rmse)
print("R² Score:", r2)


joblib.dump(model_pipeline, 'house_price_pipeline.pkl')


Mean Absolute Error: 0.20939244972068063
Mean Squared Error: 0.098913827824208
Root Mean Squared Error: 0.31450568806336077
R² Score: 0.9842103648408563


['house_price_pipeline.pkl']

In [2]:
import pandas as pd
import numpy as np
import joblib


model = joblib.load('house_price_pipeline.pkl')

# --- Sample Prediction ---
sample = pd.DataFrame({
    'property_type': ['House'],
    'location': ['Abdullah Garden'],
    'city': ['Islamabad'],
    'baths': [2],
    'purpose': ['For Sale'],
    'bedrooms': [2],
    'Area_in_Marla': [4]
})

predicted_log_price = model.predict(sample)[0]
predicted_price = np.exp(predicted_log_price)

low_price = predicted_price * 0.90
high_price = predicted_price * 1.1

print(f"Predicted Price Range: {low_price:,.0f} - {high_price:,.0f} PKR")


Predicted Price Range: 4,506,778 - 5,508,285 PKR
